In [ ]:
!pip install datasets
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

tokenizer = BertTokenizer.from_pretrained("yiyanghkust/finbert-tone")
finbert = BertForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone", num_labels=3)

from datasets import load_dataset
ds = load_dataset("kdave/Indian_Financial_News")

In [ ]:
df = ds["train"].to_pandas()

X = df[["Content"]]
y = df["Sentiment"]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
#Tokenize the inputs and pad them to a fixed length.
def preprocess_function(examples):
    max_length = 512
    return tokenizer(
        examples["Content"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

In [ ]:
train_dataset = Dataset.from_pandas(X_train.assign(labels=y_train))
val_dataset = Dataset.from_pandas(X_val.assign(labels=y_val))

# To tokenize datasets
train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

# Removing conetent now that we have info contained in the other columns
train_dataset = train_dataset.remove_columns(["Content"])
val_dataset = val_dataset.remove_columns(["Content"])

'''
train_dataset = train_dataset.rename_column("labels", "labels")
val_dataset = val_dataset.rename_column("labels", "labels")
'''

# Define a mapping for sentiment labels
label_mapping = {"Positive": 2, "Neutral": 1, "Negative": 0}

train_dataset = train_dataset.map(lambda x: {"labels": label_mapping[x["labels"]]})
val_dataset = val_dataset.map(lambda x: {"labels": label_mapping[x["labels"]]})

from datasets import Value
train_dataset = train_dataset.cast_column("labels", Value("int64"))
val_dataset = val_dataset.cast_column("labels", Value("int64"))


In [ ]:
def compute_metrics(pred):
    predictions = pred.predictions.argmax(axis=-1)
    labels = pred.label_ids

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="weighted")

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',            
    num_train_epochs=4,                
    per_device_train_batch_size=16,     # Batch size for training
    per_device_eval_batch_size=32,      
    warmup_steps=500,                   # Warmup steps for learning rate scheduler
    weight_decay=0.01,                  # Weight decay for optimization
    logging_dir='./logs',               
    logging_steps=50,                   
    evaluation_strategy="epoch",        # Evaluate the model at the end of each epoch
    save_strategy="epoch",              
    load_best_model_at_end=True,        
    metric_for_best_model="accuracy",   # Metric that is used to selecting the best model
    greater_is_better=True,             
    learning_rate=2e-5,                 # Initial learning rate
    lr_scheduler_type="linear",         # Linear learning rate scheduler
    save_total_limit=2,                 # to keep only the 2 most recent model checkpoints
    gradient_accumulation_steps=2,      # Accumulate gradients over 2 steps to handle large batches
    fp16=True,                          # Use mixed precision 
    seed=42,                         
    report_to="none"                 
)

trainer = Trainer(
    model=finbert,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
import pandas as pd
from datasets import Dataset

test_file_path = '/content/balanced_sentiment_dataset.csv'
test_df = pd.read_csv(test_file_path)

print(test_df.head())

test_df['Sentiment'].value_counts()

In [ ]:
def preprocess_function(examples):
    max_length = 512  # Maximum sequence length for FinBERT
    return tokenizer(examples["Content"], padding="max_length", truncation=True, max_length=max_length)

# This is to convert pd.dataframe to a hugging dataset
test_dataset = Dataset.from_pandas(test_df)

test_dataset = test_dataset.map(preprocess_function, batched=True)

test_dataset = test_dataset.rename_column("Sentiment", "labels")

label_mapping = {"Positive": 0, "Neutral": 1, "Negative": 2}
test_dataset = test_dataset.map(lambda x: {"labels": label_mapping[x["labels"]]})

test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

results = trainer.evaluate(test_dataset)

print("Test Set Evaluation Results:")
print(results)

In [ ]:
val_dataset

In [ ]:

test_dataset = Dataset.from_pandas(X_test.assign(labels=y_test))

In [ ]:

test_dataset = test_dataset.map(preprocess_function, batched=True)

test_dataset = test_dataset.remove_columns(["Content"])

In [ ]:

test_results = trainer.evaluate(test_dataset)


print("Test Set Evaluation Results:", test_results)

In [ ]:
test_dataset = Dataset.from_pandas(X_test.assign(labels=y_test))

# Tokenize the dataset
test_dataset = test_dataset.map(preprocess_function, batched=True)

test_dataset = test_dataset.remove_columns(["Content"])

label_mapping = {"Positive": 2, "Neutral": 1, "Negative": 0}

test_dataset = test_dataset.map(lambda x: {"labels": label_mapping[x["labels"]]})

from datasets import Value
test_dataset = test_dataset.cast_column("labels", Value("int64"))



In [ ]:
predictions = trainer.predict(test_dataset)


preds = predictions.predictions.argmax(axis=-1)

true_labels = predictions.label_ids

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

accuracy = accuracy_score(true_labels, preds)

precision, recall, f1, _ = precision_recall_fscore_support(true_labels, preds, average="weighted")


print(f"Test Accuracy: {accuracy:.4f}")

print(f"Test Precision: {precision:.4f}")

print(f"Test Recall: {recall:.4f}")

print(f"Test F1-score: {f1:.4f}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
save_path = "/content/drive/MyDrive/finbert_kdave_trained"
finbert.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
